## Franka Slide 

Check the RGB and Depth images of franka slide dataset. 

In [4]:
import h5py
import numpy as np

frankaslide_dataset = h5py.File('data/circle_m_peg_insertion_extend.hdf5', 'r')

In [22]:
print(frankaslide_dataset.keys())
print(frankaslide_dataset["demo_48"].keys())
print(frankaslide_dataset["demo_48"]["obs"].keys())
print(frankaslide_dataset["demo_48"]["actions"].shape)
print(frankaslide_dataset["demo_48"]["obs"]["agentview"].keys())
print(frankaslide_dataset["demo_48"]["obs"]["agentview"]["color"].shape)
print(frankaslide_dataset["demo_48"]["obs"]["tactile"]["finger_left"].shape)

<KeysViewHDF5 ['demo_0', 'demo_1', 'demo_10', 'demo_11', 'demo_12', 'demo_13', 'demo_14', 'demo_15', 'demo_16', 'demo_17', 'demo_18', 'demo_19', 'demo_2', 'demo_20', 'demo_21', 'demo_22', 'demo_23', 'demo_24', 'demo_25', 'demo_26', 'demo_27', 'demo_28', 'demo_29', 'demo_3', 'demo_30', 'demo_31', 'demo_32', 'demo_33', 'demo_34', 'demo_35', 'demo_36', 'demo_37', 'demo_38', 'demo_39', 'demo_4', 'demo_40', 'demo_41', 'demo_42', 'demo_43', 'demo_44', 'demo_45', 'demo_46', 'demo_47', 'demo_48', 'demo_49', 'demo_5', 'demo_50', 'demo_51', 'demo_52', 'demo_53', 'demo_54', 'demo_55', 'demo_56', 'demo_57', 'demo_58', 'demo_59', 'demo_6', 'demo_60', 'demo_61', 'demo_62', 'demo_63', 'demo_64', 'demo_65', 'demo_66', 'demo_67', 'demo_68', 'demo_69', 'demo_7', 'demo_70', 'demo_71', 'demo_72', 'demo_73', 'demo_74', 'demo_75', 'demo_76', 'demo_77', 'demo_78', 'demo_79', 'demo_8', 'demo_80', 'demo_9']>
<KeysViewHDF5 ['actions', 'done', 'language_prompts', 'obs']>
<KeysViewHDF5 ['agentview', 'delta_time',

In [6]:
# Collect all actions from the dataset and compute their norms
actions = []
for demo in frankaslide_dataset.keys():
    demo_actions = frankaslide_dataset[demo]["actions"][:][:, :3]
    actions.append(demo_actions)
actions = np.concatenate(actions, axis=0)

# Compute the norm of each action vector
actions_norm = np.linalg.norm(actions, axis=1)
print("Average norm:", actions_norm.mean(), "\nMax norm:", actions_norm.max(), "\nMin norm:", actions_norm.min(), "\nMedian norm:", np.median(actions_norm))

Average norm: 0.0007530945954053454 
Max norm: 0.0034446701657808126 
Min norm: 0.0 
Median norm: 0.0007708361952586513


In [7]:
demo_lengths = []
for demo_i in frankaslide_dataset.keys():
    length = frankaslide_dataset[demo_i]["actions"].shape[0]
    demo_lengths.append(length)
avg_demo_length = np.mean(demo_lengths)
max_demo_length = np.max(demo_lengths)
min_demo_length = np.min(demo_lengths)
print("Average demo length:", avg_demo_length)
print("Max demo length:", max_demo_length)
print("Min demo length:", min_demo_length)

Average demo length: 190.0493827160494
Max demo length: 313
Min demo length: 93


In [23]:

def downsample_franka(dataset, factor=5,
                      keys = {'agentview/color', 'agentview/depth',
                              'delta_time', 'ee_euler', 'ee_pos', 'gripper_state',
                              'gripper_width', 'joint_pos', 'joint_vel', 'tactile/finger_left'}
):
    """
    Downsample all demos in a Franka dataset by `factor` and recompute actions.

    Expected input layout per demo:
      demo_X/
        actions           [T, 7]  -> [dpos(3), deuler(3), gripper_state(1)]
        done              [T] or [T-1] (ignored; recomputed)
        language_prompts  (optional, scalar/string or array)
        obs/
          agentview/
            color       [T, ...]      (images or features)
            depth       [T, ...]      (images or features)
          delta_time      [T]           (seconds per step)
          ee_euler        [T, 3]
          ee_pos          [T, 3]
          gripper_state   [T] or [T,1]
          gripper_width   [T] or [T,1]
          joint_pos       [T, Dq]
          joint_vel       [T, Dq]
          tactile         [T, ...]      (any shape)
    """
    out = {}
    for demo_key in dataset.keys():
        g = dataset[demo_key]
        if "obs" not in g:
            continue

        obs_g = g["obs"]

        T = len(obs_g["ee_pos"])
        if T == 0:
            print(f"Warning: demo {demo_key} has zero length, skipping")
            continue

        keep = np.arange(0, T, factor)
        if keep[-1] != T - 1:
            keep = np.append(keep, T - 1)

        # build downsampled obs
        obs_out = {}
        for ok in keys:
            arr = obs_g[ok][:]

            # Handle ragged shapes like [T] vs [T,1]
            if arr.ndim == 2 and arr.shape[1] == 1:
                arr = arr.reshape(-1)

            if ok == "delta_time":
                # Sum delta_time over the windows between kept frames
                dt_list = []
                for i in range(len(keep) - 1):
                    a, b = keep[i], keep[i + 1]
                    dt_list.append(np.sum(arr[a:b]))
                # Last frame has no outgoing step
                dt_list.append(0.0)
                obs_out[ok] = np.asarray(dt_list, dtype=np.float32)
            else:
                obs_out[ok] = arr[keep]

        # recompute actions between kept frames: [dpos(3), deuler(3), gripper_state(1)]
        ee_pos_ds = np.asarray(obs_out["ee_pos"], dtype=np.float32)           # [M, 3]
        ee_euler_ds = np.asarray(obs_out["ee_euler"], dtype=np.float32)       # [M, 3]
        gripper_state_ds = np.asarray(obs_out["gripper_state"]).reshape(-1)   # [M]

        M = ee_pos_ds.shape[0]
        actions_ds = np.zeros((M, 7), dtype=np.float32)

        if M > 1:
            # Unwrap Euler angles to avoid 2π jumps before differencing
            # (assumes radians; still helps even if degrees but then set discont appropriately)
            ee_euler_unwrapped = np.unwrap(ee_euler_ds, axis=0)

            dpos = ee_pos_ds[1:] - ee_pos_ds[:-1]               # [M-1, 3]
            deul = ee_euler_unwrapped[1:] - ee_euler_unwrapped[:-1]  # [M-1, 3]
            gcmd = gripper_state_ds[1:]                         # [M-1]

            actions_ds[:-1, 0:3] = dpos
            actions_ds[:-1, 3:6] = deul
            actions_ds[:-1, 6]   = gcmd

            # Last action: no next frame -> keep zeros for deltas, set gripper to last state
            actions_ds[-1, 6] = gripper_state_ds[-1]
        else:
            # Single frame demo: zero deltas, carry gripper state
            actions_ds[0, 6] = gripper_state_ds[0]

        # recompute done flags
        done_ds = np.zeros((M,), dtype=bool)
        done_ds[-1] = True

        demo_out = {
            "obs": obs_out,
            "actions": actions_ds,
            "done": done_ds,
        }

        if "language_prompts" in g:
            lp = g["language_prompts"][()]
            demo_out["language_prompts"] = lp

        out[demo_key] = demo_out

    return out

downsampled_data = downsample_franka(frankaslide_dataset, factor=5)

In [29]:
# collect all actions from the dataset and compute their norms
actions = []
for demo in downsampled_data.keys():
    demo_actions = downsampled_data[demo]["actions"][:][:, :3]
    actions.append(demo_actions)
actions = np.concatenate(actions, axis=0)

# compute the norm of each action vector
actions_norm = np.linalg.norm(actions, axis=1)
print("Average norm:", actions_norm.mean(), "\nMax norm:", actions_norm.max(), "\nMin norm:", actions_norm.min(), "\nMedian norm:", np.median(actions_norm))

Average norm: 0.002885696 
Max norm: 0.012753366 
Min norm: 0.0 
Median norm: 0.0023632534


In [31]:
demo_lengths = []
for demo_i in downsampled_data.keys():
    length = downsampled_data[demo_i]["actions"].shape[0]
    demo_lengths.append(length)
avg_demo_length = np.mean(demo_lengths)
max_demo_length = np.max(demo_lengths)
min_demo_length = np.min(demo_lengths)
print("Average demo length:", avg_demo_length)
print("Max demo length:", max_demo_length)
print("Min demo length:", min_demo_length)

Average demo length: 39.23456790123457
Max demo length: 64
Min demo length: 20


In [32]:
def write_h5(data, out_path, compression="gzip", compression_opts=4):
    """
    Recursively write a nested dict to HDF5.
    - dict -> group
    - np.ndarray / numeric scalars -> dataset
    - str/bytes -> UTF-8 variable-length string dataset
    - list/tuple -> try to np.array(); if ragged, store as group with 0,1,2,... keys
    """
    def _to_vlen_str():
        return h5py.string_dtype(encoding="utf-8")

    def _write_node(group, key, value):
        # Allow writing the root dict into the group itself
        if key is None:
            if not isinstance(value, dict):
                raise TypeError("Root value must be a dict")
            for k, v in value.items():
                _write_node(group, k, v)
            return

        # Dict -> subgroup
        if isinstance(value, dict):
            sub = group.create_group(key)
            for k, v in value.items():
                _write_node(sub, k, v)
            return

        # Numpy arrays
        if isinstance(value, np.ndarray):
            group.create_dataset(key, data=value,
                                compression=compression,
                                compression_opts=compression_opts)
            return

        # Strings / bytes
        if isinstance(value, (str, bytes, np.bytes_)):
            group.create_dataset(key, data=value, dtype=_to_vlen_str())
            return

        # Numeric scalars
        if isinstance(value, Number) or (hasattr(value, "shape") and value.shape == ()):
            group.create_dataset(key, data=value)
            return

        # Lists/tuples: try dense array, else fall back to subgroup with indices
        if isinstance(value, (list, tuple)):
            try:
                arr = np.asarray(value)
                group.create_dataset(key, data=arr,
                                    compression=compression,
                                    compression_opts=compression_opts)
            except Exception:
                sub = group.create_group(key)
                for i, v in enumerate(value):
                    _write_node(sub, str(i), v)
            return

        # Fallback: try numpy array conversion
        try:
            arr = np.asarray(value)
            group.create_dataset(key, data=arr,
                                compression=compression,
                                compression_opts=compression_opts)
            return
        except Exception as e:
            raise TypeError(f"Unsupported type for key '{key}': {type(value)}") from e

    with h5py.File(out_path, "w") as f:
        _write_node(f, None, data)

write_h5(downsampled_data, "data/circle_m_peg_insertion_extend_downsampled5.hdf5")